# India Petrochemical Trade Analysis & Forecast (2026-2027)## Project OverviewThis notebook provides a comprehensive analysis of India's petrochemical trade,including data exploration, correlation analysis, predictive modeling, and12-month forecasts under multiple geopolitical scenarios.**Key Context (June 2026):**- Brent Crude: ~$90/barrel (Strait of Hormuz tensions)- USD/INR: ~95.5 (record low)- India GDP: 7.7% (FY2026)- Oil Import Dependency: 85-87%

In [ ]:
import warningswarnings.filterwarnings('ignore')import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom pathlib import Pathplt.style.use('seaborn-v0_8-whitegrid')sns.set_palette('husl')%matplotlib inlineDATA_DIR = Path('data/processed')FIG_DIR = Path('figures')print('Setup complete.')

## 1. Data Overview

In [ ]:
# Load processed datasetsdatasets = {}for f in DATA_DIR.glob('*.csv'):    datasets[f.stem] = pd.read_csv(f)    print(f'Loaded: {f.stem} ({len(datasets[f.stem])} rows)')print(f'\nTotal datasets: {len(datasets)}')

## 2. Petrochemical Trade Balance

In [ ]:
tb = datasets.get('petrochemical_trade_balance')if tb is not None:    print(tb.to_string(index=False))else:    print('Trade balance data not available')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))# Exports vs Importsax = axes[0, 0]if tb is not None:    x = range(len(tb))    ax.bar(x, tb['total_export_value']/1000, label='Exports', color='#2ecc71', alpha=0.8)    ax.bar(x, -tb['total_import_value']/1000, label='Imports', color='#e74c3c', alpha=0.8)    ax.set_xticks(x)    ax.set_xticklabels(tb['fiscal_year'], rotation=45)    ax.set_ylabel('Value (thousands crore ₹)')    ax.set_title('Petrochemical Exports vs Imports')    ax.legend()    ax.axhline(y=0, color='black', linewidth=0.5)# Trade Balanceax = axes[0, 1]if tb is not None:    ax.plot(tb['fiscal_year'], tb['trade_balance']/1000, 'o-', color='#3498db', linewidth=2)    ax.fill_between(range(len(tb)), tb['trade_balance']/1000, alpha=0.3, color='#3498db')    ax.set_xticks(range(len(tb)))    ax.set_xticklabels(tb['fiscal_year'], rotation=45)    ax.set_ylabel('Trade Balance (thousands crore ₹)')    ax.set_title('Petrochemical Trade Balance')    ax.axhline(y=0, color='red', linestyle='--', linewidth=1)plt.tight_layout()plt.savefig(FIG_DIR / 'notebook_01_trade_balance.pdf', dpi=150, bbox_inches='tight')plt.show()

## 3. Macroeconomic Correlations

In [ ]:
# Load macro datamacro = {}for name in ['wti_crude', 'usd_inr', 'gdp_growth', 'fdi_inflows', 'cpi_inflation']:    path = DATA_DIR / f'macro_{name}.csv'    if path.exists():        df = pd.read_csv(path)        df['date'] = pd.to_datetime(df['date'])        macro[name] = df.set_index('date')[[name]].resample('M').mean()if macro:    merged = pd.DataFrame()    for name, df in macro.items():        merged = merged.join(df, how='outer') if not merged.empty else df    merged = merged.dropna()    print(f'Merged macro data: {len(merged)} months')    print(merged.describe())else:    print('No macro data available')

In [ ]:
if 'merged' in dir() and len(merged) > 10:    fig, axes = plt.subplots(1, 2, figsize=(16, 6))    # Correlation heatmap    corr = merged.corr()    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,                ax=axes[0], square=True, linewidths=0.5)    axes[0].set_title('Macro Indicator Correlations')    # Rolling correlation    if 'wti_crude' in merged.columns and 'usd_inr' in merged.columns:        rolling_corr = merged['wti_crude'].rolling(12).corr(merged['usd_inr'])        axes[1].plot(merged.index, rolling_corr, color='#9b59b6', linewidth=2)        axes[1].axhline(y=0, color='black', linestyle='--', linewidth=0.5)        axes[1].set_title('Rolling 12M Correlation: Crude vs USD/INR')        axes[1].set_ylabel('Correlation')    plt.tight_layout()    plt.savefig(FIG_DIR / 'notebook_02_correlations.pdf', dpi=150, bbox_inches='tight')    plt.show()

## 4. Predictive Model Forecasts

In [ ]:
fc_path = DATA_DIR.parent / 'results' / '03_forecasts.csv'if fc_path.exists():    forecasts = pd.read_csv(fc_path, index_col=0, parse_dates=True)    print('Available forecasts:')    print(forecasts.columns.tolist())    print(forecasts.tail(12))else:    print('Run 03_models.py first to generate forecasts.')

In [ ]:
metrics_path = DATA_DIR.parent / 'results' / '02_model_metrics.csv'if metrics_path.exists():    metrics = pd.read_csv(metrics_path)    print('\nModel Comparison:')    print(metrics.to_string(index=False))        fig, ax = plt.subplots(figsize=(10, 5))    colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12']    ax.barh(metrics['model'], metrics['mae'], color=colors[:len(metrics)])    ax.set_xlabel('Mean Absolute Error (lower = better)')    ax.set_title('Model Performance Comparison')    plt.tight_layout()    plt.savefig(FIG_DIR / 'notebook_03_model_comparison.pdf', dpi=150, bbox_inches='tight')    plt.show()

## 5. 2026-2027 Scenario Analysis

In [ ]:
scenario_path = DATA_DIR.parent / 'results' / '04_scenario_forecasts.csv'if scenario_path.exists():    scenarios = pd.read_csv(scenario_path)    scenarios['date'] = pd.to_datetime(scenarios['date'])        fig, axes = plt.subplots(1, 2, figsize=(16, 6))    colors = {'base_case': '#3498db', 'bull_case': '#2ecc71', 'bear_case': '#e74c3c'}        for scenario_id in scenarios['scenario'].unique():        data = scenarios[scenarios['scenario'] == scenario_id]        axes[0].plot(data['date'], data['import_index'], 'o-',                     color=colors.get(scenario_id, 'gray'), linewidth=2, label=scenario_id)        axes[1].plot(data['date'], data['trade_balance'], 'o-',                     color=colors.get(scenario_id, 'gray'), linewidth=2, label=scenario_id)        axes[0].set_title('Import Index Forecast')    axes[0].set_ylabel('Index')    axes[0].legend()    axes[1].set_title('Trade Balance Forecast')    axes[1].set_ylabel('Balance Index')    axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1)    axes[1].legend()        plt.tight_layout()    plt.savefig(FIG_DIR / 'notebook_04_scenarios.pdf', dpi=150, bbox_inches='tight')    plt.show()

## 6. Key Conclusions### Findings1. **Structural Deficit**: India runs a persistent trade deficit in petrochemicals2. **Crude Price Sensitivity**: Petrochemical imports are highly sensitive to crude oil prices3. **Currency Impact**: INR depreciation amplifies import costs in domestic terms4. **GDP Correlation**: Industrial growth drives petrochemical demand### 2026 Outlook- **Base Case**: Petrochemical imports remain elevated due to crude prices- **Bull Case**: Peace dividend could reduce import costs by 15-20%- **Bear Case**: Escalation could push import costs up by 20-30%### Policy Recommendations1. Build strategic petrochemical reserves2. Diversify crude sourcing away from Middle East3. Invest in domestic petrochemical capacity4. Implement currency hedging for importers5. Shift toward specialty chemicals with higher value-add